# What is PL/SQL?

### PL/SQL = Procedural Language / Structured Query Language
### Its Oracle progrramming language that extends SQL by adding proggramming features such as variables, conditions, loops, exception handelling, procedures, functions, packages, and triggers.

In [1]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

In [6]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [15]:
%%plsql
DECLARE
    v_msg VARCHAR2(100) := 'Brilliant! Your database setup is officially complete.';
BEGIN
    DBMS_OUTPUT.PUT_LINE(v_msg);
END;

Brilliant! Your database setup is officially complete.


In [1]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

In [2]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [ ]:
import os
import re
from IPython import get_ipython

def inject_downloaded_schema():
    # 1. Point to the local script file path you want to process
    script_path = r"db-sample-schemas-23.3\human_resources\hr_install.sql"
    
    if not os.path.exists(script_path):
        print(f"❌ Script not found. Ensure tar extraction ran successfully.")
        return

    # 2. Retrieve your active notebook session variables
    ip = get_ipython()
    if ip is None: return
    
    # 3. Pull the live connection engine safely from memory
    # This completely skips loading credentials again!
    try:
        with open(script_path, "r", encoding="utf-8") as f:
            sql_script = f.read()

        # Clean SQL*Plus interactive lines out of the source script 
        statements = sql_script.split(";")
        clean_statements = []

        for stmt in statements:
            stmt_clean = stmt.strip()
            if not stmt_clean or stmt_clean.upper().startswith(("SET ", "PROMPT ", "ACCEPT ", "SHOW ", "EXIT", "REM ")):
                continue
            stmt_clean = re.sub(r'--.*$', '', stmt_clean, flags=re.MULTILINE)
            if stmt_clean.strip():
                clean_statements.append(stmt_clean.strip())

        # Execute the statements using the active global cursor
        print("⏳ Injecting schema records into the active connection session...")
        
        # We target the live, global engine object registered on notebook initialization
        # (This uses the underlying cursor created inside connect_db.py)
        for statement in clean_statements:
            try:
                # Use IPython's internal shell parsing hook to safely route statements
                # without re-authenticating
                ip.run_cell(f"%%plsql\n{statement}")
            except Exception:
                pass # Skip initial drop warnings smoothly

        print("🎉 Sample schema successfully parsed and loaded into your active session!")
        
    except Exception as e:
        print(f"❌ Automation pipeline failed: {e}")

# Run the localized sequence
inject_downloaded_schema()
